In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path
import cv2
import json
import pickle
from sklearn.metrics import roc_curve, auc, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import load_model
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def load_gan_models(generator_path, discriminator_path):
    """Load pre-trained GAN models from H5 files."""
    try:
        generator = load_model(generator_path)
        discriminator = load_model(discriminator_path)
        
        return generator, discriminator
    except Exception as e:
        raise
class TumorClassifier:
    def __init__(self, n_clusters=3):  # 3 tumor types
        self.scaler = StandardScaler()
        self.kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        self.cluster_labels = {}  # Map clusters to tumor types
        self.fitted = False
    
    def fit(self, features, labels):
        """Fit the classifier using feature vectors and their corresponding labels."""

        scaled_features = self.scaler.fit_transform(features)
        

        self.kmeans.fit(scaled_features)
        
        cluster_assignments = {}
        for cluster in range(self.kmeans.n_clusters):
            mask = self.kmeans.labels_ == cluster
            cluster_labels = [label for label, m in zip(labels, mask) if m]
            most_common = max(set(cluster_labels), key=cluster_labels.count)
            cluster_assignments[cluster] = most_common
        
        self.cluster_labels = cluster_assignments
        self.fitted = True
    
    def predict(self, features):
        """Predict tumor type from feature vector."""
        if not self.fitted:
            raise ValueError("Classifier must be fitted before predicting")
        

        scaled_features = self.scaler.transform(features)
        
        clusters = self.kmeans.predict(scaled_features)
        
        return [self.cluster_labels[c] for c in clusters]

class GANAnomalyDetector:
    def __init__(self, generator, discriminator, image_shape=(256, 256, 1), latent_dim=100):
        self.generator = generator
        self.discriminator = discriminator
        self.image_shape = image_shape
        self.latent_dim = latent_dim
        self.classifier = TumorClassifier()
        

        self.feature_extractor = tf.keras.Model(
            inputs=self.discriminator.inputs,
            outputs=self.discriminator.layers[-2].output
        )
        
    def compute_latent_vector(self, image):
        """Find the optimal latent vector for reconstruction."""
        z = tf.Variable(tf.random.normal([1, self.latent_dim]))
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
        
        @tf.function
        def optimize_step():
            with tf.GradientTape() as tape:
                generated = self.generator(z, training=False)
                loss = tf.reduce_mean(tf.square(generated - image))
            
            gradients = tape.gradient(loss, [z])
            optimizer.apply_gradients(zip(gradients, [z]))
            return loss
        
        
        return z
    
    def save_models(self, save_dir):
        """Save all models and necessary data."""
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        

        self.generator.save(save_dir / 'generator.h5')
        self.discriminator.save(save_dir / 'discriminator.h5')
        

        if self.classifier.fitted:
            with open(save_dir / 'classifier.pkl', 'wb') as f:
                pickle.dump(self.classifier, f)
        
        config = {
            'image_shape': self.image_shape,
            'latent_dim': self.latent_dim
        }
        with open(save_dir / 'config.json', 'w') as f:
            json.dump(config, f)
    
    @classmethod
    def load_models(cls, load_dir):
        """Load saved models and configuration."""
        load_dir = Path(load_dir)
        

        with open(load_dir / 'config.json', 'r') as f:
            config = json.load(f)
        

        generator = load_model(load_dir / 'generator.h5')
        discriminator = load_model(load_dir / 'discriminator.h5')
        
        detector = cls(
            generator=generator,
            discriminator=discriminator,
            image_shape=tuple(config['image_shape']),
            latent_dim=config['latent_dim']
        )
        
        classifier_path = load_dir / 'classifier.pkl'
        if classifier_path.exists():
            with open(classifier_path, 'rb') as f:
                detector.classifier = pickle.load(f)
        
        return detector

    def preprocess_image(self, image_path):
        """Load and preprocess a single image."""
        img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (self.image_shape[0], self.image_shape[1]))
        img = (img.astype('float32') - 127.5) / 127.5
        img = np.expand_dims(img, axis=-1)
        return img

    def extract_features(self, image):
        """Extract features from image using discriminator."""
        if len(image.shape) == 3:
            image = tf.expand_dims(image, 0)
        
        features = self.feature_extractor(image, training=False)
        return features.numpy().reshape(1, -1)

    def compute_anomaly_score(self, image):
        """Compute anomaly score and extract features."""
        if len(image.shape) == 3:
            image = tf.expand_dims(image, 0)
            
        z = self.compute_latent_vector(image)
        reconstructed = self.generator(z, training=False)
        reconstruction_error = tf.reduce_mean(tf.square(image - reconstructed))
        
        real_features = self.feature_extractor(image, training=False)
        fake_features = self.feature_extractor(reconstructed, training=False)
        feature_error = tf.reduce_mean(tf.square(real_features - fake_features))
        
        real_score = self.discriminator(image, training=False)
        fake_score = self.discriminator(reconstructed, training=False)
        discrimination_error = tf.reduce_mean(tf.square(real_score - fake_score))
        
        total_score = (
            0.4 * reconstruction_error +
            0.3 * feature_error +
            0.3 * discrimination_error
        )
        
        return {
            'total_score': total_score.numpy(),
            'reconstruction_error': reconstruction_error.numpy(),
            'feature_error': feature_error.numpy(),
            'discrimination_error': discrimination_error.numpy(),
            'features': real_features.numpy().reshape(-1)
        }

    def train_classifier(self, data_dir):
        """Train the tumor classifier using the dataset."""
        features = []
        labels = []
        
        data_dir = Path(data_dir)
        for class_dir in data_dir.iterdir():
            if class_dir.is_dir() and class_dir.name != 'notumor':
                print(f"Processing {class_dir.name} for training...")
                for img_path in class_dir.glob('*.jpg'):
                    try:
                        img = self.preprocess_image(img_path)
                        score_dict = self.compute_anomaly_score(img)
                        features.append(score_dict['features'])
                        labels.append(class_dir.name)
                    except Exception as e:
                        print(f"Error processing {img_path}: {e}")
        
        features = np.array(features)
        self.classifier.fit(features, labels)
        print("Classifier training completed!")

def evaluate_dataset(detector, data_dir, threshold=None):
    """Evaluate the anomaly detector and classifier on a dataset."""
    data_dir = Path(data_dir)
    results = []
    labels = []
    features = []
    
    print("\nProcessing images...")
    for class_dir in data_dir.iterdir():
        if class_dir.is_dir():
            is_normal = class_dir.name == 'notumor'
            print(f"\nProcessing {class_dir.name} images...")
            
            for img_path in class_dir.glob('*.jpg'):
                try:
                    img = detector.preprocess_image(str(img_path))
                    scores = detector.compute_anomaly_score(img)
                    
                    results.append({
                        'path': str(img_path),
                        'class': class_dir.name,
                        'scores': scores,
                        'is_normal': is_normal
                    })
                    labels.append(0 if is_normal else 1)
                    if not is_normal:
                        features.append(scores['features'])
                    
                except Exception as e:
                    print(f"Error processing {img_path}: {e}")
    
    print("\nComputing metrics...")
    
    if threshold is None:
        scores = [r['scores']['total_score'] for r in results]
        fpr, tpr, thresholds = roc_curve(labels, scores)
        roc_auc = auc(fpr, tpr)
        optimal_idx = np.argmax(tpr - fpr)
        threshold = thresholds[optimal_idx]
    
    if features:
        features = np.array(features)
        tumor_predictions = detector.classifier.predict(features)
    else:
        tumor_predictions = []
    
    predictions = []
    tumor_idx = 0
    for r in results:
        is_anomaly = r['scores']['total_score'] > threshold
        pred_dict = {
            'path': r['path'],
            'class': r['class'],
            'is_anomaly': is_anomaly,
            'score': r['scores']['total_score'],
            'correct': is_anomaly != r['is_normal']
        }
        
        if is_anomaly and not r['is_normal']:
            pred_dict['predicted_type'] = tumor_predictions[tumor_idx]
            pred_dict['type_correct'] = tumor_predictions[tumor_idx] == r['class']
            tumor_idx += 1
        
        predictions.append(pred_dict)
    
    return predictions, threshold

def visualize_results(predictions, save_dir):
    """Visualize and save detection and classification results."""
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    plt.figure(figsize=(12, 6))
    data = []
    labels = []
    for p in predictions:
        data.append(p['score'])
        labels.append(p['class'])
    
    sns.violinplot(x=labels, y=data)
    plt.title('Anomaly Scores Distribution by Class')
    plt.ylabel('Anomaly Score')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(save_dir / 'anomaly_scores.png')
    plt.close()
    
    tumor_preds = [(p['predicted_type'], p['class']) 
                   for p in predictions 
                   if 'predicted_type' in p]
    if tumor_preds:
        true_labels = [t[1] for t in tumor_preds]
        pred_labels = [t[0] for t in tumor_preds]
        classes = sorted(set(true_labels))
        
        cm = confusion_matrix(true_labels, pred_labels, labels=classes)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes)
        plt.title('Tumor Classification Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig(save_dir / 'confusion_matrix.png')
        plt.close()

def main():
    model_dir = '/kaggle/working/saved_models'
    test_dir = '/kaggle/input/brain-tumor-mri-dataset/Testing'
    train_dir = '/kaggle/input/brain-tumor-mri-dataset/Training'
    results_dir = '/kaggle/working/results'
    
    generator_path = '/kaggle/input/braingan/generator_epoch_9000.h5'
    discriminator_path = '/kaggle/input/braingan/discriminator_epoch_9000.h5'
    save_dir = '/kaggle/working'
    print("Loading models...")
    generator, discriminator = load_gan_models(generator_path, discriminator_path)
    
    detector = GANAnomalyDetector(
        generator=generator,
        discriminator=discriminator,
        image_shape=(256, 256, 1),
        latent_dim=100
    )
    
    print("\nTraining tumor classifier...")
    detector.train_classifier(train_dir)
    
    print("\nSaving models...")
    detector.save_models(model_dir)
    
    print("\nEvaluating on test set...")
    predictions, threshold = evaluate_dataset(detector, test_dir)
    
    correct = sum(1 for p in predictions if p['correct'])
    total = len(predictions)
    print(f"\nOverall Tumor Detection Accuracy: {correct/total:.2%}")
    
    type_correct = sum(1 for p in predictions if 'type_correct' in p and p['type_correct'])
    type_total = sum(1 for p in predictions if 'type_correct' in p)
    if type_total > 0:
        print(f"Tumor Classification Accuracy: {type_correct/type_total:.2%}")
    
    print("\nPer-class Results:")
    for class_name in ['notumor', 'meningioma', 'glioma', 'pituitary']:
        class_preds = [p for p in predictions if p['class'] == class_name]
        if class_preds:
            correct = sum(1 for p in class_preds if p['correct'])
            print(f"{class_name} detection accuracy: {correct/len(class_preds):.2%}")
    
    print("\nGenerating visualizations...")
    visualize_results(predictions, results_dir)
    
    print(f"\nResults and visualizations saved to {results_dir}")
    print(f"Models saved to {model_dir}")

if __name__ == "__main__":
    main()